<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/stage_07_0x_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage_07_XX – <MODEL_NAME> (Seq2Seq 29×8 → 29×1)

Esta notebook entrena y evalúa el modelo **<MODEL_NAME>** para el problema seq2seq:
- **Entrada:** (29 × 8)
- **Salida:** (29 × 1) con \(\Delta pts_h\) por minuto (h = 60 o 90)

**Output:** métricas y predicciones out-of-sample guardadas como artefactos para el **Stage_08**.

## **1. Imports + paths**

In [ ]:
import os
import json
import time
import random
from pathlib import Path

import numpy as np

# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# -----------------------------
# Paths del proyecto
# -----------------------------
PROJECT_ROOT = Path("..").resolve()          # notebooks/ -> raíz
DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports" / "stage_07"
MODEL_DIR = PROJECT_ROOT / "models" / "stage_07"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## **2. Reproducibilidad**

In [ ]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **3. Configuración**

In [ ]:
# -----------------------------
# Config general del stage
# -----------------------------
HORIZON = 60           # 60 o 90
SEQ_LEN = 29
N_FEATURES = 8

MODEL_NAME = "<MODEL_NAME>"   # ej: "naive", "mlp", "lstm", "tcn", "transformer", "tft"

# Umbrales para métricas económicas (filtro de oportunidad)
THETA = 20.0           # umbral mínimo para disparar señal (en puntos)
DELTA_OP = 84.0        # umbral de oportunidad "explotable" (ej: delta_target_p70 de stage_03a)

# Rutas de artefactos por horizonte/modelo
OUT_DIR = REPORT_DIR / f"h{HORIZON}" / MODEL_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_MODEL_DIR = MODEL_DIR / f"h{HORIZON}" / MODEL_NAME
OUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)


## **4. Importar métricas comunes desde .py**

In [ ]:
import sys
sys.path.append(str(PROJECT_ROOT))

from src.neural_profit.metrics.seq2seq import compute_seq2seq_metrics
from src.neural_profit.metrics.econ_filters import compute_opportunity_filter_metrics


## **6. Carga de data windows**

In [ ]:
def load_npz_windows(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Carga X e Y desde un .npz estándar."""
    data = np.load(path)
    X = data["X"]
    Y = data["Y"]
    return X, Y

# Ejemplo de naming (ajuste si su stage_06 usa otra convención exacta)
# data/windows/scaled/windows_{split}_{h}_z.npz
train_path = DATA_DIR / "windows" / "scaled" / f"windows_train_{HORIZON}_z.npz"
valid_path = DATA_DIR / "windows" / "scaled" / f"windows_valid_{HORIZON}_z.npz"
test_path  = DATA_DIR / "windows" / "scaled" / f"windows_test_{HORIZON}_z.npz"

X_train, Y_train = load_npz_windows(train_path)
X_valid, Y_valid = load_npz_windows(valid_path)
X_test,  Y_test  = load_npz_windows(test_path)

X_train.shape, Y_train.shape, X_valid.shape, Y_valid.shape, X_test.shape, Y_test.shape

## **7. Sanity Check**

In [ ]:
def sanity_check(X: np.ndarray, Y: np.ndarray, name: str) -> None:
    assert X.ndim == 3, f"{name}: X debe ser 3D (n, seq, feat)"
    assert Y.ndim in (2, 3), f"{name}: Y debe ser 2D o 3D (n, seq) o (n, seq, 1)"
    assert X.shape[1] == SEQ_LEN and X.shape[2] == N_FEATURES, f"{name}: shape X inesperado"
    assert Y.shape[1] == SEQ_LEN, f"{name}: shape Y inesperado"
    assert np.isfinite(X).all(), f"{name}: X contiene NaN/inf"
    assert np.isfinite(Y).all(), f"{name}: Y contiene NaN/inf"

sanity_check(X_train, Y_train, "train")
sanity_check(X_valid, Y_valid, "valid")
sanity_check(X_test,  Y_test,  "test")

print("OK - shapes y valores finitos.")

## **8. Definición del modelo — placeholder**

Variante A: baseline naive (ejemplo listo)

In [ ]:
def predict_naive_last_value(X: np.ndarray) -> np.ndarray:
    """
    Baseline simple:
    predice una secuencia constante igual al último valor de una feature elegida.
    IMPORTANTE: Ajuste esto al baseline que defina (por ejemplo 0, o último delta observado si existe).
    """
    n = X.shape[0]
    # Aquí asumimos que no hay delta pasado en X, entonces baseline = 0
    y_pred = np.zeros((n, SEQ_LEN), dtype=float)
    return y_pred

y_pred_valid = predict_naive_last_value(X_valid)
y_pred_test  = predict_naive_last_value(X_test)

## **9. Métricas ML**

In [ ]:
ml_valid = compute_seq2seq_metrics(Y_valid, y_pred_valid, compute_r2=True)
ml_test  = compute_seq2seq_metrics(Y_test,  y_pred_test,  compute_r2=True)

ml_valid, ml_test

## **10. Métricas económicas como filtro**

In [ ]:
econ_valid = compute_opportunity_filter_metrics(
    y_true=Y_valid,
    y_pred=y_pred_valid,
    delta_op=DELTA_OP,
    theta=THETA,
)

econ_test = compute_opportunity_filter_metrics(
    y_true=Y_test,
    y_pred=y_pred_test,
    delta_op=DELTA_OP,
    theta=THETA,
)

econ_valid, econ_test

## **11. Guardar artefactos para Stage_08**

In [ ]:
def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# Guardar métricas
save_json({"split": "valid", "horizon": HORIZON, "model": MODEL_NAME, **ml_valid}, OUT_DIR / "metrics_ml_valid.json")
save_json({"split": "test",  "horizon": HORIZON, "model": MODEL_NAME, **ml_test},  OUT_DIR / "metrics_ml_test.json")

save_json({"split": "valid", "horizon": HORIZON, "model": MODEL_NAME, **econ_valid}, OUT_DIR / "metrics_econ_valid.json")
save_json({"split": "test",  "horizon": HORIZON, "model": MODEL_NAME, **econ_test},  OUT_DIR / "metrics_econ_test.json")

# Guardar predicciones OOS (para análisis posterior)
np.savez_compressed(
    OUT_DIR / "pred_test.npz",
    y_true=np.asarray(Y_test),
    y_pred=np.asarray(y_pred_test),
)

print("OK - artefactos guardados en:", OUT_DIR)

## **12. Resumen**

- **Métricas ML (test):** MAE, RMSE, DA_last, R2
- **Métricas económicas como filtro (test):** Precision, Opportunity_Recall, Coverage
- Artefactos guardados en `reports/stage_07/h{H}/<model_name>/`